# Lab 03: Attention from Scratch
**Course:** Large Language Models and Query Engineering
**Instructor:** Dr. Akram Fatayer
**Week 3 · Session B (lab)** · Supports CLO1

<a href="https://colab.research.google.com/github/akramfatayer/zuj-llm/blob/main/labs/lab-03-attention/Lab_03_Attention_from_Scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

---

## Plan for today (75 minutes)

| Time | What |
|---|---|
| 5 min | Brief: the attention formula, and the shapes you will meet |
| 35 min | Parts A–D: attention, word order, scaling, the causal mask and multi-head attention, in small tensors you can read |
| 25 min | Part E: the same attention inside Qwen2.5, in English and Arabic |
| 10 min | Part F: reflect, and read the project brief |

By the end of this lab you will be able to:

1. Implement scaled dot-product attention, **softmax(QKᵀ / √d) V**, and check it against PyTorch's own.
2. Explain why attention needs position information, and why the scores are divided by √d.
3. Add a causal mask and prove the model cannot peek at later tokens.
4. Build a multi-head self-attention layer and match it to `nn.MultiheadAttention`.
5. Extract attention weights from a real model and find a head with a clear job.

Cells marked `# TODO` are yours to complete; the check cell after each prints ✅ or tells you what to fix. **Stuck?** Open the 💡 hints under the cell one at a time, after trying for a few minutes. An error that mentions `NoneType` almost always means a TODO line still says `None`. Parts A–D run on any laptop; Part E loads the 1.5-billion-parameter lab model.

## Part 0 — Setup

In [ ]:
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !pip -q install -U "transformers>=4.45"

In [ ]:
import math
import os

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
torch.set_printoptions(precision=2, sci_mode=False)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", torch.cuda.get_device_name(0) if device == "cuda" else "CPU (fine for Parts A–D)")

## Part A — Attention by hand (10 min)

Every token asks a question (its **query**, q), every token advertises what it contains (its **key**, k), and every token carries the information it will hand over (its **value**, v). Attention compares one query with all keys, turns the scores into weights with a softmax, and returns the weighted average of the values:

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^\top}{\sqrt{d}}\right) V$$

Below are four tokens with 3-dimensional queries, keys and values. Small integers, so you can follow the numbers. The cell works through the steps for **one** query, the last token.

In [ ]:
tokens = ["Ali", "reads", "a", "book"]
Q = torch.tensor([[1., 0, 1], [0, 2, 0], [1, 1, 0], [2, 0, 1]])
K = torch.tensor([[2., 0, 1], [0, 1, 0], [0, 0, 1], [1, 2, 0]])
V = torch.tensor([[1., 0, 0], [0, 1, 0], [0, 0, 1], [1, 1, 1]])

q = Q[3]                              # the query of "book"
scores = K @ q                        # one dot product per key
scaled = scores / math.sqrt(3)        # divide by √d (Part C explains why)
weights = torch.softmax(scaled, dim=0)
output = weights @ V                  # weighted average of the values

for tok, s, w in zip(tokens, scores, weights):
    print(f"{tok:>6}: score {s:4.1f}   weight {w:.2f}  {'█' * round(w.item() * 30)}")
print("output for 'book':", output, " (weights sum to", round(weights.sum().item(), 3), ")")

**Question:** "book" gives most of its weight to "Ali". Which numbers in Q and K cause that? What would you change in `K` to make "book" attend mostly to "reads"?

*Write your answer here.*

### Exercise A — All queries at once

Now do every query in one matrix product. Complete `attention()`. It takes tensors whose **last two** dimensions are (tokens, features), so the same code later works for a batch and for many heads. `mask` is optional: where it is `False`, the token may not be attended to (Part D needs this).

In [ ]:
def attention(Q, K, V, mask=None):
    """Scaled dot-product attention. Returns (output, weights)."""
    d = Q.shape[-1]
    # TODO 1: scores between every query and every key, divided by √d   (hint: K.transpose(-2, -1))
    scores = None
    if mask is not None:
        scores = scores.masked_fill(~mask, float("-inf"))  # a weight of exactly 0 after the softmax
    # TODO 2: softmax over the keys (the last dimension)
    weights = None
    # TODO 3: weighted average of the values
    output = None
    return output, weights

<details>
<summary>💡 Hint 1: follow the shapes</summary>

With n tokens and d features, `Q`, `K` and `V` are n × d. The step-by-step cell above did one query with `K @ q`; for all queries at once:

- `Q @ K.transpose(-2, -1)` is n × n: row i holds query i against every key. Divide it by `math.sqrt(d)`.
- The softmax must make every **row** of weights sum to 1, so it runs over the last dimension.
- `weights @ V` is n × d again: one weighted average of the values per query.

</details>

<details>
<summary>💡 Hint 2: the functions</summary>

`torch.softmax(x, dim=-1)` normalises over the last dimension. Use `transpose(-2, -1)` rather than `.T`, so that the same code still works when a batch or a head dimension is added in front.

</details>

In [ ]:
# Check: the small example, then random batches against PyTorch's own implementation
out, W = attention(Q, K, V)
assert torch.allclose(out[3], output), "row 3 should match the step-by-step result above"
assert torch.allclose(W.sum(-1), torch.ones(4)), "every row of weights should sum to 1"
q, k, v = (torch.randn(2, 5, 8) for _ in range(3))
assert torch.allclose(attention(q, k, v)[0], F.scaled_dot_product_attention(q, k, v), atol=1e-5)
print("✅ attention() matches torch.nn.functional.scaled_dot_product_attention")

plt.figure(figsize=(3.6, 3))
plt.imshow(W, cmap="Oranges", vmin=0, vmax=1)
plt.xticks(range(4), tokens)
plt.yticks(range(4), tokens)
plt.xlabel("key (attended to)")
plt.ylabel("query (attending)")
plt.colorbar()
plt.title("Attention weights")
plt.tight_layout()
plt.show()

## Part B — Attention cannot see word order (5 min)

In a real layer, Q, K and V are computed from the token embeddings with three learned matrices. Nothing in the formula says *where* a token is. Here are "dog bites man" and "man bites dog", built from the same word vectors.

In [ ]:
d_model = 8
vocab = {w: torch.randn(d_model) for w in ["dog", "bites", "man"]}
W_q, W_k, W_v = (torch.randn(d_model, d_model) for _ in range(3))


def self_attention(X):
    out, _ = attention(X @ W_q, X @ W_k, X @ W_v)
    return out


s1 = ["dog", "bites", "man"]
s2 = ["man", "bites", "dog"]
X1 = torch.stack([vocab[w] for w in s1])
X2 = torch.stack([vocab[w] for w in s2])
out1, out2 = self_attention(X1), self_attention(X2)
print("output for 'bites' is the same in both sentences:", torch.allclose(out1[1], out2[1], atol=1e-5))
print("output for 'dog' is the same, just moved:        ", torch.allclose(out1[0], out2[2], atol=1e-5))

The fix is to give every position its own signal. The original transformer **adds** a fixed vector per position (sines and cosines of different frequencies). Run the cell to see the outputs become different.

In [ ]:
def sinusoidal_positions(n, d):
    """The position vectors of Vaswani et al. (2017): one row per position."""
    pos = torch.arange(n).unsqueeze(1)
    freq = torch.exp(-math.log(10000.0) * torch.arange(0, d, 2) / d)
    P = torch.zeros(n, d)
    P[:, 0::2] = torch.sin(pos * freq)
    P[:, 1::2] = torch.cos(pos * freq)
    return P


P = sinusoidal_positions(3, d_model)
out1, out2 = self_attention(X1 + P), self_attention(X2 + P)
print("with positions, 'bites' still the same?", torch.allclose(out1[1], out2[1], atol=1e-5))

plt.figure(figsize=(6, 1.8))
plt.imshow(sinusoidal_positions(40, 32).T, cmap="RdBu", aspect="auto")
plt.xlabel("position")
plt.ylabel("dimension")
plt.title("Sinusoidal position vectors: every column is different")
plt.tight_layout()
plt.show()

**Question:** Qwen2.5 and most current models do not add position vectors to the input. They use **rotary position embeddings (RoPE)**, which rotate each query and key by an angle that grows with its position. Why does rotating *both* q and k make their dot product depend on the *distance* between two tokens rather than on their absolute positions? (One or two sentences; the lecture slide on RoPE helps.)

*Write your answer here.*

## Part C — Why divide by √d? (5 min)

The dot product of two random vectors grows with their dimension d. Large scores make the softmax put almost all the weight on one key, and then the gradients through the other keys vanish during training. Run the cell.

In [ ]:
print(f"{'d':>5}  {'std of q·k':>11}  {'max weight, unscaled':>21}  {'max weight, ÷√d':>16}")
for d in [4, 16, 64, 256, 1024]:
    q, keys = torch.randn(d), torch.randn(10, d)
    s = keys @ q
    print(f"{d:>5}  {torch.randn(2000, d).mul(torch.randn(2000, d)).sum(1).std():>11.1f}  "
          f"{torch.softmax(s, 0).max():>21.2f}  {torch.softmax(s / math.sqrt(d), 0).max():>16.2f}")

**Question:** what is the standard deviation of q·k close to, as a function of d? Why does dividing by √d keep the softmax useful at every size? (Qwen2.5-1.5B uses d = 128 per head.)

*Write your answer here.*

## Part D — No peeking: the causal mask, and multi-head attention (15 min)

A decoder is trained to predict each next token, so position *i* may only attend to positions 0…*i*. Otherwise it could copy the answer from the future.

### Exercise D1 — The causal mask

Complete `causal_mask(n)`: an n × n boolean tensor that is `True` where a query (row) may attend to a key (column).

In [ ]:
def causal_mask(n):
    # TODO: True on and below the diagonal, False above it   (hint: torch.tril)
    return None

<details>
<summary>💡 Hint</summary>

`torch.tril(x)` keeps the entries of `x` on and below the diagonal and sets the others to zero:

```
torch.tril(torch.ones(3, 3))   →   [[1, 0, 0],
                                    [1, 1, 0],
                                    [1, 1, 1]]
```

The check wants `True` and `False` rather than 1 and 0: create the ones with `dtype=torch.bool`.

</details>

In [ ]:
# Check 1: the shape of the mask
m = causal_mask(4)
assert m.dtype == torch.bool and m.shape == (4, 4), "the mask should be a 4 × 4 bool tensor"
assert m[3].all() and not m[0, 1:].any(), "the last row sees everything; the first row only itself"
print(m.int())

# Check 2: changing the LAST token must not change any earlier output
x = torch.randn(6, d_model)
y = x.clone()
y[-1] = torch.randn(d_model)
a, Wa = attention(x @ W_q, x @ W_k, x @ W_v, causal_mask(6))
b, _ = attention(y @ W_q, y @ W_k, y @ W_v, causal_mask(6))
assert torch.allclose(a[:-1], b[:-1], atol=1e-5), "earlier positions changed: the mask lets them see the future"
assert torch.allclose(a, F.scaled_dot_product_attention(x @ W_q, x @ W_k, x @ W_v, is_causal=True), atol=1e-4)
print("✅ causal mask works: earlier tokens cannot see the last one, and the result matches PyTorch's is_causal=True")

### Exercise D2 — Multi-head attention

One attention computation can only produce one weighted average per token. **Multi-head attention** runs *h* of them side by side, each on its own slice of d_model / h features, so different heads can follow different relationships. The outputs are put back side by side and mixed by an output matrix W_o.

Complete `forward()`. The heads are just an extra batch dimension, so your `attention()` already handles them.

In [ ]:
class MultiHeadSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0
        self.h, self.d_head = n_heads, d_model // n_heads
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x):  # x: (batch, n, d_model)
        b, n, d_model = x.shape

        def split(t):  # (b, n, d_model) -> (b, h, n, d_head)
            return t.view(b, n, self.h, self.d_head).transpose(1, 2)

        q, k, v = split(self.W_q(x)), split(self.W_k(x)), split(self.W_v(x))
        # TODO 1: causal attention in every head at once, using attention() and causal_mask()
        out, weights = None, None
        # TODO 2: put the heads back side by side, (b, h, n, d_head) -> (b, n, d_model), then apply W_o
        out = None
        return out, weights

<details>
<summary>💡 Hint 1: TODO 1</summary>

After `split()`, `q`, `k` and `v` are (b, h, n, d_head). Your `attention()` only works on the last two dimensions, so one call, `attention(q, k, v, mask)`, runs every head of every sequence at once. The mask is `causal_mask(n)`; add `.to(x.device)` so it also works on a GPU.

</details>

<details>
<summary>💡 Hint 2: TODO 2</summary>

Undo `split()` step by step, in reverse: `out.transpose(1, 2)` gives (b, n, h, d_head), then `.reshape(b, n, d_model)` puts the h heads side by side. Finally mix them with the output projection, `self.W_o(...)`, and return that together with the weights.

</details>

In [ ]:
# Check: same weights in PyTorch's nn.MultiheadAttention must give the same output
mine = MultiHeadSelfAttention(d_model=32, n_heads=4)
ref = nn.MultiheadAttention(32, 4, bias=False, batch_first=True)
with torch.no_grad():
    ref.in_proj_weight.copy_(torch.cat([mine.W_q.weight, mine.W_k.weight, mine.W_v.weight]))
    ref.out_proj.weight.copy_(mine.W_o.weight)
x = torch.randn(2, 7, 32)
out, weights = mine(x)
expected, _ = ref(x, x, x, attn_mask=~causal_mask(7), need_weights=False)  # PyTorch's mask: True = blocked
assert out.shape == (2, 7, 32) and weights.shape == (2, 4, 7, 7), f"shapes: {tuple(out.shape)}, {tuple(weights.shape)}"
assert torch.allclose(out, expected, atol=1e-5), "output differs from nn.MultiheadAttention"
print("✅ your multi-head layer matches nn.MultiheadAttention")
print(f"parameters: {sum(p.numel() for p in mine.parameters()):,} = 4 × d_model²")

## Part E — Attention inside a real model (25 min)

Qwen2.5-1.5B stacks 28 decoder blocks. Each block has a multi-head attention layer like yours, followed by a feed-forward network. We load it with the plain ("eager") attention code, because the fast fused kernels never build the weight matrix and so cannot show it to us.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers.utils import logging as hf_logging

hf_logging.disable_progress_bar()
MODEL_ID = os.environ.get("LAB_MODEL", "Qwen/Qwen2.5-1.5B-Instruct")
# bfloat16 on a GPU: in float16 this model's plain attention code overflows and returns NaN
dtype = torch.bfloat16 if device == "cuda" and torch.cuda.is_bf16_supported() else torch.float32
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
try:
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=dtype, attn_implementation="eager")
except TypeError:  # transformers older than 4.56 calls it torch_dtype
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=dtype, attn_implementation="eager")
model = model.to(device).eval()

cfg = model.config
print(f"{MODEL_ID}: {cfg.num_hidden_layers} layers × {cfg.num_attention_heads} heads")
print(f"d_model = {cfg.hidden_size}, d_head = {cfg.hidden_size // cfg.num_attention_heads}, "
      f"key/value heads = {cfg.num_key_value_heads} (grouped-query attention: heads share keys and values)")

In [ ]:
@torch.no_grad()
def attention_maps(text):
    """All attention weights for `text`: a (layers, heads, n, n) tensor, plus the tokens."""
    enc = tokenizer(text, return_tensors="pt").to(device)
    out = model(**enc, output_attentions=True)
    attn = torch.stack(out.attentions)[:, 0].float().cpu()
    return attn, [tokenizer.decode([i]) for i in enc.input_ids[0]]


def plot_head(attn, tokens, layer, head, ax=None):
    """Heatmap of one head. Arabic tokens are shown by number (matplotlib cannot join Arabic letters)."""
    ax = ax or plt.subplots(figsize=(5, 4.2))[1]
    arabic = any("\u0600" <= ch <= "\u06ff" for t in tokens for ch in t)
    labels = [str(i) for i in range(len(tokens))] if arabic else [t.strip() or repr(t) for t in tokens]
    ax.imshow(attn[layer, head], cmap="Oranges", vmin=0, vmax=1)
    ax.set_xticks(range(len(tokens)), labels, rotation=90, fontsize=8)
    ax.set_yticks(range(len(tokens)), labels, fontsize=8)
    ax.set_title(f"layer {layer}, head {head}", fontsize=10)
    return ax


english = "The animal didn't cross the street because it was too tired."
attn, toks = attention_maps(english)
print("attention tensor:", tuple(attn.shape), " (layers, heads, query, key)")
print(list(enumerate(toks)))

fig, axes = plt.subplots(1, 3, figsize=(15, 4.6))
for ax, (layer, head) in zip(axes, [(0, 0), (13, 5), (27, 3)]):
    plot_head(attn, toks, layer, head, ax)
plt.tight_layout()
plt.show()

Every map is lower-triangular: that is the causal mask you wrote. In many heads most of the weight lands on the **first token**. This "attention sink" is a known habit of trained models: when a head has nothing useful to look at, it parks its weight on the first position.

### Exercise E1 — Find the previous-token head

Some heads have one clear job. A **previous-token head** sends almost all of each token's weight to the token just before it. Complete `previous_token_score()`: for every layer and head, the average weight from position *i* to position *i − 1*, over *i* = 1 … n − 1.

In [ ]:
def previous_token_score(attn):
    """attn: (layers, heads, n, n) -> (layers, heads)"""
    n = attn.shape[-1]
    i = torch.arange(1, n)
    # TODO: the weights attn[..., i, i - 1] averaged over the positions i
    return None

<details>
<summary>💡 Hint 1: indexing with two index tensors</summary>

Indexing a matrix with two index tensors picks one entry per pair of indices. On a 4 × 4 matrix `A`, with `i = torch.arange(1, 4)`:

```python
A[i, i - 1]   # [A[1, 0], A[2, 1], A[3, 2]]: the entries just below the diagonal
```

`attn[..., i, i - 1]` does the same in the last two dimensions, for every layer and head at once.

</details>

<details>
<summary>💡 Hint 2: the average</summary>

`attn[..., i, i - 1]` has shape (layers, heads, n − 1). Average over the last dimension with `.mean(dim=-1)` to get one score per (layer, head).

</details>

In [ ]:
scores = previous_token_score(attn)
assert scores is not None and scores.shape == attn.shape[:2], "one score per (layer, head)"
assert (scores >= 0).all() and (scores <= 1.01).all(), "scores are averages of weights, so between 0 and 1"
best = divmod(int(scores.argmax()), scores.shape[1])
print(f"✅ best previous-token head: layer {best[0]}, head {best[1]} (score {scores.max():.2f})")

fig, axes = plt.subplots(1, 2, figsize=(13, 4.4), gridspec_kw={"width_ratios": [1.4, 1]})
im = axes[0].imshow(scores.T, cmap="Oranges", vmin=0, vmax=1, aspect="auto")
axes[0].set_xlabel("layer")
axes[0].set_ylabel("head")
axes[0].set_title("previous-token score of every head")
fig.colorbar(im, ax=axes[0])
plot_head(attn, toks, *best, axes[1])
plt.tight_layout()
plt.show()

### Exercise E2 — Does the same head do the same job in Arabic?

Run the best head on an Arabic sentence. The tokens are printed with their numbers, because the heatmap labels them by number.

In [ ]:
arabic = "لم يعبر الحيوان الشارع لأنه كان متعباً جداً."
attn_ar, toks_ar = attention_maps(arabic)
print(list(enumerate(toks_ar)))
scores_ar = previous_token_score(attn_ar)
print(f"same head on Arabic: score {scores_ar[best]:.2f}   (best Arabic head: layer "
      f"{divmod(int(scores_ar.argmax()), scores_ar.shape[1])[0]}, head {divmod(int(scores_ar.argmax()), scores_ar.shape[1])[1]}, "
      f"score {scores_ar.max():.2f})")
plot_head(attn_ar, toks_ar, *best)
plt.show()

### Exercise E3 — Where does "it" look?

In the first sentence, "it" refers to "the animal". Change "tired" to "wide" and "it" refers to "the street". `where_does_it_look()` prints, for a few layers, the three tokens a word attends to most (averaged over the heads, ignoring the first token, the sink).

**Before you run it:** predict whether "it" will look at different tokens in the two sentences.

In [ ]:
def where_does_it_look(text, word, layers=(4, 8, 12, 16, 20, 24)):
    attn, toks = attention_maps(text)
    i = toks.index(word)
    print(f"{word.strip()!r} in: {text}")
    for layer in layers:
        w = attn[layer, :, i, 1:].mean(0)  # average over heads, skip the sink
        top = torch.topk(w, 3)
        print(f"  layer {layer:>2}: " + ",  ".join(f"{toks[j + 1].strip()!r} {p:.2f}" for p, j in zip(top.values, top.indices)))
    print()


tired = "The animal didn't cross the street because it was too tired."
wide = "The animal didn't cross the street because it was too wide."
where_does_it_look(tired, " it")
where_does_it_look(wide, " it")

The two printouts are identical. "it" comes *before* "tired" and "wide", and the causal mask stops it from seeing them, so at the moment the model reads "it" the two sentences are the same. The model can only resolve "it" later, at the last word. Look there instead:

In [ ]:
where_does_it_look(tired, " tired")
where_does_it_look(wide, " wide")

**Questions:**

1. Does "tired" attend more to "animal" and "wide" more to "street", in any layer? What does this say about *where* in a sentence a decoder can resolve a pronoun?
2. Attention weights show where information *can* flow, not *why* the model decides. Give one reason why a high attention weight is not a full explanation of the model's behaviour.

*Write your answers here.*

## Part F — Reflect (5 min)

1. **Exit question.** In one sentence each: what do the query, the key and the value of a token do?
2. Your multi-head layer has 4 × d_model² parameters, whatever the number of heads. So what does adding more heads change?
3. Qwen2.5-1.5B has 12 query heads but only 2 key/value heads. What does sharing keys and values save when the model generates long answers? (We return to this in week 4.)

*Write your answers here.*

---

## The team project starts now

Read the project brief: [project/README.md](https://github.com/akramfatayer/zuj-llm/blob/main/project/README.md). Form a team of 3–4 and register it on Moodle **by the week 4 lab**. Phase 1, the proposal and data plan, is due in week 7.

**Before next week:** finish the interactive module [Inside Attention](https://akramfatayer.github.io/courses/llm/learn/2) and paste its completion code into Moodle. Read Raschka, *Build a Large Language Model (From Scratch)*, chapters 4–5, and skim Kaplan et al. (2020) on scaling laws.